# Galerkin experiments with layered and discontinuous materials

This notebook solves a BSK model or a verification problem with a known answer.
Choose **1, 2, 3, 5, 7, or 10 interfaces**, or enter your own material table.
The last layer always extends to infinity. Both uniform and smoothly graded
layers are supported.

**First use:** install the updated project once with
`python -m pip install -e ".[notebook]"` from the project directory. Open this
notebook with a kernel from that same environment. Then choose **Run All**.
Changing the inputs below does not require another installation. After updating
the project code, restart the kernel before Run All to load the new helpers.

The main workflow performs three solves: your selected experiment and two
known-answer checks. The separately controlled larger and extreme BSK cases
are also enabled in the current settings; their switches are shown beside
their own resolutions and Newton targets. Separate cells show relative field errors and
optional resolution-convergence tables for the two known-answer problems. Each solve declares its own
resolution `N`. Larger sweeps, resolution checks, arbitrary functions, and a
separate high-precision example have their own enable switches below.


## The model and the quantities we plot

We solve
$$
\left[\epsilon(x)(u'-\delta^2u''')\right]'=f(x,u),\qquad x>0,
$$
with a prescribed wall potential and wall third derivative. The potential
vanishes at infinity. The mixed Galerkin solver also approximates
$$
w=-u'',
$$ 
the curvature field. The model flux is
$J=\epsilon(u'+\delta^2w')$.

Coordinates and potentials use the **dimensionless scaling of the model**;
inputs here are not automatically converted from metres or volts. In BSK,
$\delta$ is the correlation-length parameter, $\sigma$ controls the spatial
width of the wall-related source, $\gamma$ controls the nonlinear denominator,
and $\mu$ controls its amplitude. The implementation used in this project is
$$
f(x,u)=\frac{\sinh u-\tfrac12\mu g_\sigma(x)e^u}
 {1+2\gamma\sinh^2(u/2)},\qquad
 g_\sigma(x)=\frac{e^{-x^2/(2\sigma^2)}}{\sqrt{2\pi}}.
$$
This preserves the package's Gaussian convention, **without a $1/\sigma$
prefactor**. A finite plotting window does not truncate the half-line model.


In [1]:
from pathlib import Path
import os
from matplotlib import pyplot as plt
from overscreening_breakdown import (
    BSKProblem, FluxProblem, LayeredMaterial, preset_material, Coefficient,
    ContinuationOptions, ScreenedLinearReference,
)
from overscreening_breakdown.manufactured import layered_transmission_case
from overscreening_breakdown.notebook_examples.discontinuous_galerkin import (
    run_defined_problem, report_experiment, report_verification, material_table,
    plot_material, plot_material_gallery, plot_experiment, plot_errors,
    plot_convergence, compare_resolution,
    report_error_table, report_convergence_table,
    homotopy_stage_rows, report_homotopy_stages, report_homotopy_comparison,
)
from overscreening_breakdown.notebook_examples.cases import CaseArtifacts, export_cases
from overscreening_breakdown.notebook_examples.homotopy_plotting import (
    plot_homotopy_path, plot_homotopy_comparison,
)

%matplotlib inline
notebook_cases = {}
# Safe empty state until the separate BSK case cells set their own switches.
run_large_bsk = False
run_extreme_bsk_mpmath = False
bsk_large_experiment = None
bsk_extreme_experiment = None
bsk_large_figures = {}
bsk_extreme_figures = {}
output_directory = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")).expanduser().resolve() / "galerkin-discontinuous"
print("Saved results will be placed in:", output_directory)


Saved results will be placed in: /data/Programs/overscreening_breakdown/examples/notebooks/notebook-output/galerkin-discontinuous


## 1. Your experiment — edit this cell

- `interface_count`: choose **1, 2, 3, 5, 7, or 10**.
- `material_style`: `"constant"` for uniform layers, or `"graded"` for smooth
  variation inside each layer followed by a jump at its boundary.
- `model`: `"BSK"`, `"manufactured_linear"`, or `"manufactured_nonlinear"`.
- `accuracy`: `"standard"` for NumPy arithmetic, `"strict_integration"` for
  tighter integration checks, or `"high_precision"` for 50-digit mpmath arithmetic.
  These profiles never choose or change `N` or the Newton absolute tolerance.
  Set both beside each solve.

Quoted decimal inputs retain their spelling for high-precision calculations.
The BSK wall data are $u(0=$ `wall_potential` and $u'''(0)=0$.
The known-answer models choose their own exact wall data; BSK-specific wall,
roughness, and denominator settings do not change those verification problems.

Leave `layers=None` to use the preset. To enter your own material, replace it
with the commented table: each row contains **start position, epsilon value**.
The first row starts at zero; the final row extends to infinity. The table
itself determines the number of interfaces when supplied.


In [ ]:
interface_count = 3
material_style = "constant"       # "constant" or "graded"
model = "BSK"                     # or "BSK_large", "manufactured_linear", "manufactured_nonlinear"
accuracy = "strict_integration"             # "standard", "strict_integration", "high_precision"

wall_potential = 1.0
delta = 1
gamma = 0.5
sigma = 2
mu = 0.1
verbosity = 1                     # 0: quiet; 1: main phases; 2: iteration progress

layers = None
# For a custom three-interface material, replace the line above with:
layers = [
    (0,   0.1),
    (0.5, 10.0),
    (1.5, 0.2),
    (3.0, 22.8),
    (6.0, 1.8),
]

# Display only: change plot_x_max to "50" for a wider view.
plot_x_min = "0"
plot_x_max = "10"

# History axes count Newton iterations and quadrature orders, not physical x.
iteration_xlim = None       # Example: (0, 50)
quadrature_xlim = None      # Example: (0, 128)


## 2. Check the material before solving

The helper supplies the bounds, wall trace, and interface locations. No
coefficient derivative is needed by the BSK solver. In a graded layer, the
coefficient starts at the table value and gradually approaches 1.1 times that
value, with grading length 1 in the same coordinate units. The printed table
shows both sides of every jump. Values exactly at an interface use its right
trace; isolated point values do not change the weak equations.


In [ ]:
material = (preset_material(interface_count, kind=material_style) if layers is None
            else LayeredMaterial(layers, kind=material_style))
print(material_table(material, accuracy=accuracy))
material_figure = plot_material(material, accuracy=accuracy, x_min=plot_x_min, x_max=plot_x_max)
plt.show()


## 2a. Define the BSK physical problems

The next cells contain the physical definitions. Edit the constructors,
initial guesses, or source callbacks here. Each local definition function
captures its inputs and returns a problem object; the same function is reused
when the optional sweep changes the material.

`BSKProblem` supplies the BSK source displayed above. Here all parameters,
the half-line domain, boundary conditions, coefficient, and initial guess are
explicit. Boundary rows list derivative orders 0 through 4; `None` leaves an
order unspecified. The two rows refer to the wall and infinity respectively.


In [ ]:
def define_bsk_problem(epsilon, *, wall_potential, delta, gamma, sigma, mu):
    def initial_guess(x, a):
        return a.scalar(wall_potential) * a.exp(-x)

    return BSKProblem(
        delta=delta, gamma=gamma, sigma=sigma, mu=mu,
        epsilon=epsilon, domain=(0, "inf"),
        boundary_conditions=(
            (wall_potential, None, None, "0", None),
            ("0", None, None, None, None),
        ),
        initial_guess=initial_guess,
    )

bsk_problem = define_bsk_problem(
    material.coefficient, wall_potential=wall_potential,
    delta=delta, gamma=gamma, sigma=sigma, mu=mu,
)

bsk_problem_large = define_bsk_problem(
    material.coefficient, wall_potential=wall_potential*10,
    delta=delta, gamma=gamma, sigma=sigma, mu=mu,
)

bsk_problem_large_large = define_bsk_problem(
    material.coefficient, wall_potential=wall_potential*1000,
    delta=delta, gamma=gamma, sigma=sigma, mu=mu,
)


## 2b. Define the manufactured linear physical problem

The catalog supplies only the known transmission reference used here. Its
`u(x, a)` and `flux_dx(x, a)` evaluate $u_*$ and the analytic ordinary
within-layer derivative $J_*'$, using the selected arithmetic. The exact
reference satisfies flux continuity at the interfaces.

The physical source and its derivative are written explicitly below:
$$
f(x,u)=J_*'(x)+(u-u_*),\qquad f_u=1.
$$
`FluxProblem` takes these **unscaled** callbacks. No homotopy factor or
normalization belongs in the problem definition.


In [ ]:
def define_linear_problem(material, *, delta):
    reference = layered_transmission_case(material, nonlinear=False, delta=delta)

    def f(x, u, a):
        return reference.flux_dx(x, a) + (u - reference.u(x, a))

    def f_u(x, u, a):
        return a.scalar(1)

    def initial_guess(x, a):
        return a.exp(-x)

    problem = FluxProblem(
        delta=delta, epsilon=material.coefficient, domain=(0, "inf"),
        f=f, f_u=f_u,
        boundary_conditions=(
            (1, None, None, -6, None),
            (0, None, None, None, None),
        ),
        initial_guess=initial_guess,
    )
    return problem, reference

linear_problem, linear_case = define_linear_problem(material, delta=delta)


## 2c. Define the manufactured nonlinear physical problem

The nonlinear reaction strength `beta_nonlinear` is explicit. The callbacks
implement
$$
f(x,u)=J_*'(x)+(u-u_*)+\beta(u^3-u_*^3),\qquad
f_u=1+3\beta u^2.
$$
All arithmetic inside callbacks uses `a`, so the same definition supports
NumPy and mpmath. The reference construction is independent of the numerical
resolution. If you replace the source with a different physical model, supply
its derivative too. Set `case=None` when solving a problem without a known
reference; known-answer comparisons then do not apply to that result.


In [ ]:
beta_nonlinear = 0.125

def define_nonlinear_problem(material, *, delta, beta):
    reference = layered_transmission_case(material, nonlinear=True, delta=delta, beta=beta)

    def f(x, u, a):
        exact = reference.u(x, a)
        return (reference.flux_dx(x, a) + (u - exact)
                + a.scalar(beta) * (u**3 - exact**3))

    def f_u(x, u, a):
        return a.scalar(1) + 3 * a.scalar(beta) * u**2

    def initial_guess(x, a):
        return a.exp(-x)

    problem = FluxProblem(
        delta=delta, epsilon=material.coefficient, domain=(0, "inf"),
        f=f, f_u=f_u,
        boundary_conditions=(
            (1, None, None, -6, None),
            (0, None, None, None, None),
        ),
        initial_guess=initial_guess,
    )
    return problem, reference

nonlinear_problem, nonlinear_case = define_nonlinear_problem(
    material, delta=delta, beta=beta_nonlinear,
)


## 3. Run the selected experiment

Choose the already defined problem using `model` in the input cell, then set `N_selected` and
`newton_atol_selected` explicitly in the next cell. `run_defined_problem` solves
the supplied object; the model label is used for reporting. Galerkin `N` controls the
**global approximation spaces**: `N-1` coefficients for the potential correction
and `N` for the independent curvature field, giving **`2*N-1` unknowns**.
For example, `N=12` gives 23 unknowns. Choose an integer `N >= 2`.
All material layers share these spaces; the interface count does not set `N`.

The Newton absolute tolerance is a positive residual target. The solver stops
when the algebraic Euclidean norm of the physical discrete residual satisfies
$$
\|R_h(z;0)\|_2 \le \texttt{newton\_atol}.
$$
The relative tolerance is zero. This measures satisfaction of the discrete
equations; physical field error and integration error have their own checks.
Keep decimal targets quoted, for example `"1e-10"` for NumPy or `"1e-35"`
with the 50-digit mpmath profile. Changing profiles does not change this target.

Three statements have different meanings:

- **Equations solved:** the nonlinear discrete equations met their target.
- **Numerical integration verified:** refining the integration rule changed
  the residual and Jacobian by less than their integration targets.
- **Resolution sensitivity:** how the physical fields change when more basis
  functions are used. This is not checked by the first two statements.

The numerical presets are starting points, not promises of a fixed number of
accurate physical digits. Reference continuation is available automatically if
Newton needs recovery. The progress output reports actual recovery use.


In [ ]:
N_selected = 20
newton_atol_selected = 1e-12
selected_figures = {}
notebook_cases.pop("selected", None)

selected_problem, selected_case = {
    "BSK": (bsk_problem, None),
    "BSK_large": (bsk_problem_large, None),
    "BSK_large_large": (bsk_problem_large_large, None),
    "manufactured_linear": (linear_problem, linear_case),
    "manufactured_nonlinear": (nonlinear_problem, nonlinear_case),
}[model]
experiment = run_defined_problem(
    selected_problem, material=material, case=selected_case, model=model,
    N=N_selected, newton_atol=newton_atol_selected,
    accuracy=accuracy, verbosity=verbosity,
)
bsk_result = experiment if model == "BSK" else None
print(report_experiment(experiment))
if not experiment.solution.converged:
    raise RuntimeError("This experiment is not verified. Inspect the report before using its fields.")


## 4. Read the physical profiles and numerical history

Dotted vertical guides mark material interfaces. Potential and curvature are
continuous in the mathematical transmission problem. The **exact physical
flux is also continuous**, but the current global Galerkin approximation can
show finite-resolution jumps in its computed flux. Its pieces are drawn
separately. Do not interpret a numerical pointwise flux jump as an imposed
interface source; inspect refinement, weak balance, and integrated field error.

The convergence plot uses residual divided by the actual stage target. Values
below one satisfy that stage's target. Changes of equation parameter or
integration plan are separate curve segments. Only the physical endpoint
$q=0$, with integration verified, is a successful final solve.

### Display controls

Edit `plot_x_min` and `plot_x_max` in the experiment settings to choose the
physical x window for profiles, material previews, and error plots. The default is `[0, 8]`; use `plot_x_max="50"` to view up to 50.

These settings change drawing and sampling only. Solver inputs, stored
solutions, quadrature checks, and full-domain physical error norms stay the same.
Redraw the plotting calls using the stored results after changing the window.
`iteration_xlim` limits the Newton-iteration axis and `quadrature_xlim` limits
the quadrature-order axis; use `None` for automatic history limits.


In [ ]:
selected_fields_figure = plot_experiment(experiment, x_min=plot_x_min, x_max=plot_x_max)
selected_checks_figure = plot_convergence(experiment, iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
selected_figures = {
    "fields": selected_fields_figure,
    "convergence": selected_checks_figure,
    "material": material_figure,
}
notebook_cases["selected"] = CaseArtifacts("selected", experiment, figures=selected_figures)
plt.show()


### Separate BSK experiment: ten times the wall potential

This case has its own `run_large_bsk`, `N_large`, and `newton_atol_large`.
Its current settings retain the larger-potential experiment already used here.
Disabling it skips both its solve and plots and leaves the selected `model`
and `experiment` unchanged. Its figures and exported data belong to the
`bsk-large` case bundle.


In [ ]:
run_large_bsk = True
N_large = 50
newton_atol_large = 1e-11
bsk_large_experiment = None
bsk_large_figures = {}
notebook_cases.pop("bsk-large", None)
if run_large_bsk:
    bsk_large_experiment = run_defined_problem(
        bsk_problem_large, material=material, case=None, model="BSK_large",
        N=N_large, newton_atol=newton_atol_large,
        accuracy=accuracy, verbosity=verbosity,
    )
    print(report_experiment(bsk_large_experiment))
    if not bsk_large_experiment.solution.converged:
        raise RuntimeError("The larger-wall-potential experiment is not verified. Inspect its report before using its fields.")
else:
    print("Larger BSK experiment not run. Set run_large_bsk=True to enable it.")


In [ ]:
if run_large_bsk and bsk_large_experiment is not None:
    bsk_large_fields_figure = plot_experiment(bsk_large_experiment, x_min=plot_x_min, x_max=plot_x_max)
    bsk_large_checks_figure = plot_convergence(bsk_large_experiment, iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
    bsk_large_figures = {
        "fields": bsk_large_fields_figure,
        "convergence": bsk_large_checks_figure,
        "material": material_figure,
    }
    notebook_cases["bsk-large"] = CaseArtifacts(
        "bsk-large", bsk_large_experiment, figures=bsk_large_figures,
    )
    plt.show()
else:
    notebook_cases.pop("bsk-large", None)


### Separate BSK experiment: one thousand times the wall potential

`run_extreme_bsk_mpmath` controls this independent mpmath calculation. Set
`N_extreme` and `newton_atol_extreme` beside its solve. Its model label is
`BSK_large_large`, matching its problem definition. Its figures and exports
belong to `bsk-extreme` and never replace the selected result.

The later manufactured high-precision example has its original, separate
`run_high_precision_example` switch and numerical settings.


In [ ]:
run_extreme_bsk_mpmath = True
N_extreme = 35
newton_atol_extreme = 1e-30
bsk_extreme_experiment = None
bsk_extreme_figures = {}
notebook_cases.pop("bsk-extreme", None)
if run_extreme_bsk_mpmath:
    bsk_extreme_experiment = run_defined_problem(
        bsk_problem_large_large, material=material, case=None, model="BSK_large_large",
        N=N_extreme, newton_atol=newton_atol_extreme,
        accuracy="high_precision", verbosity=verbosity,
    )
    print(report_experiment(bsk_extreme_experiment))
else:
    print("Extreme BSK mpmath experiment not run. Set run_extreme_bsk_mpmath=True to enable it.")


In [ ]:
if run_extreme_bsk_mpmath and bsk_extreme_experiment is not None:
    bsk_extreme_fields_figure = plot_experiment(bsk_extreme_experiment, x_min=plot_x_min, x_max=30)
    bsk_extreme_checks_figure = plot_convergence(bsk_extreme_experiment, iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
    bsk_extreme_figures = {
        "fields": bsk_extreme_fields_figure,
        "convergence": bsk_extreme_checks_figure,
        "material": material_figure,
    }
    notebook_cases["bsk-extreme"] = CaseArtifacts(
        "bsk-extreme", bsk_extreme_experiment, figures=bsk_extreme_figures,
    )
    plt.show()
else:
    notebook_cases.pop("bsk-extreme", None)


## 5. Preview all six interface counts

This gallery is cheap: it plots material definitions without solving six new
problems. Change `interface_count` above and rerun from the material preview to
study another case. The optional sweep near the end solves every preset.


In [ ]:
gallery_figure = plot_material_gallery(kind=material_style, x_min=plot_x_min, x_max=plot_x_max)
selected_figures["material-gallery"] = gallery_figure
notebook_cases["selected"] = CaseArtifacts("selected", experiment, figures=selected_figures)
plt.show()


## 6. Verification with a known answer — linear RHS

These checks construct the reference field and forcing together. The exact
potential, its first two derivatives, and its physical flux satisfy
the transmission conditions at every interface, including cases with nonzero flux. Its third derivative can jump.
The known wall values are $u_*(0)=1$, $u_*'''(0)=-6$, and $u_*(\infty)=0$.
The chosen `delta` and material are retained.

Writing $J_*=\epsilon(u_*'-\delta^2u_*''')$, the linear check uses
$$
f(x,u)=J_*'(x)+(u-u_*).
$$
The plots show pointwise errors, and the report estimates physical
$L^2(dx)$ errors using an independently refined, interface-split rule.


In [ ]:
notebook_cases.pop("linear-check", None)
N_linear = 30
newton_atol_linear = 1e-10

linear_check = run_defined_problem(
    linear_problem, material=material, case=linear_case, model="manufactured_linear",
    N=N_linear, newton_atol=newton_atol_linear,
    accuracy=accuracy, verbosity=verbosity,
)
print(report_experiment(linear_check))
print(report_verification(linear_check))
if not linear_check.solution.converged:
    raise RuntimeError("The linear verification did not pass its numerical checks.")
linear_error_figure = plot_errors(linear_check, x_min=plot_x_min, x_max=plot_x_max)
linear_figures = {"errors": linear_error_figure}
notebook_cases["linear-check"] = CaseArtifacts(
    "linear-check", linear_check, figures=linear_figures,
)
plt.show()


## 6a. Relative errors at the current resolution

The curves above show pointwise absolute error. This separate table measures
errors over the full physical half-line using the $L^2(dx)$ norm:
$$
E_{\rm abs}=\|v_N-v_*\|_{L^2(dx)},\qquad
E_{\rm rel}(\%)=100\frac{\|v_N-v_*\|_{L^2(dx)}}{\|v_*\|_{L^2(dx)}}.
$$
Here $v$ is the potential $u$, auxiliary field $w$, or flux $J$.
The table uses an independently refined, interface-split integration rule.
Relative error is undefined when the reference norm is zero. A small Newton
residual verifies the discrete equations; this table measures physical field
error against the known answer. This cell performs no new solve.


In [ ]:
linear_relative_error_table = report_error_table(linear_check)
print(linear_relative_error_table)


## 6b. Optional resolution-convergence study

Set `run_linear_convergence=True` to run the explicitly listed resolutions.
Each row reuses `linear_problem` exactly as defined above, with the same material, `delta`, arithmetic
profile, and Newton absolute tolerance. Edit the list and quoted target below.
The larger resolutions can take appreciably longer, especially with mpmath.

The tables show absolute and percentage-relative physical errors, the Newton
residual, and whether both equation and integration checks passed. Unverified
rows are provisional. Inspect the trend across several resolutions; individual
errors can oscillate, and no exponential convergence rate is assumed for
interface problems. A one-row list reports errors but cannot show a trend.


In [ ]:
run_linear_convergence = True
N_values_linear = [8, 12, 20, 30, 48]
newton_atol_linear_convergence = "1e-10"
linear_convergence_results = ()
if run_linear_convergence:
    linear_convergence_results = tuple(
        run_defined_problem(
            linear_problem, material=material, case=linear_case,
            model="manufactured_linear", N=N,
            newton_atol=newton_atol_linear_convergence,
            accuracy=accuracy, verbosity=verbosity,
        )
        for N in N_values_linear
    )
    print(report_convergence_table(linear_convergence_results))
else:
    print("Linear resolution study not run. Set run_linear_convergence=True to enable it.")


## 7. Verification with a known answer — nonlinear RHS

The nonlinear check uses the same exact fields, with
$$
f(x,u)=J_*'(x)+(u-u_*)+\frac18(u^3-u_*^3),\qquad
f_u=1+\frac38u^2.
$$
Both checks include genuine transmission behaviour. Numerical error need not
vanish merely because the nonlinear residual is small. Jumps reduce global
regularity, so an exponential refinement rate is not assumed.


In [ ]:
notebook_cases.pop("nonlinear-check", None)
N_nonlinear = 20
newton_atol_nonlinear = 1e-10

nonlinear_check = run_defined_problem(
    nonlinear_problem, material=material, case=nonlinear_case, model="manufactured_nonlinear",
    N=N_nonlinear, newton_atol=newton_atol_nonlinear,
    accuracy=accuracy, verbosity=verbosity,
)
print(report_experiment(nonlinear_check))
print(report_verification(nonlinear_check))
if not nonlinear_check.solution.converged:
    raise RuntimeError("The nonlinear verification did not pass its numerical checks.")
nonlinear_error_figure = plot_errors(nonlinear_check, x_min=plot_x_min, x_max=plot_x_max)
nonlinear_figures = {"errors": nonlinear_error_figure}
notebook_cases["nonlinear-check"] = CaseArtifacts(
    "nonlinear-check", nonlinear_check, figures=nonlinear_figures,
)
plt.show()


## 7a. Relative errors at the current resolution

The curves above show pointwise absolute error. This separate table measures
errors over the full physical half-line using the $L^2(dx)$ norm:
$$
E_{\rm abs}=\|v_N-v_*\|_{L^2(dx)},\qquad
E_{\rm rel}(\%)=100\frac{\|v_N-v_*\|_{L^2(dx)}}{\|v_*\|_{L^2(dx)}}.
$$
Here $v$ is the potential $u$, auxiliary field $w$, or flux $J$.
The table uses an independently refined, interface-split integration rule.
Relative error is undefined when the reference norm is zero. A small Newton
residual verifies the discrete equations; this table measures physical field
error against the known answer. This cell performs no new solve.


In [ ]:
nonlinear_relative_error_table = report_error_table(nonlinear_check)
print(nonlinear_relative_error_table)


## 7b. Optional resolution-convergence study

Set `run_nonlinear_convergence=True` to run the explicitly listed resolutions.
Each row reuses `nonlinear_problem` exactly as defined above, with the same material, `delta`, arithmetic
profile, and Newton absolute tolerance. Edit the list and quoted target below.
The larger resolutions can take appreciably longer, especially with mpmath.

The tables show absolute and percentage-relative physical errors, the Newton
residual, and whether both equation and integration checks passed. Unverified
rows are provisional. Inspect the trend across several resolutions; individual
errors can oscillate, and no exponential convergence rate is assumed for
interface problems. A one-row list reports errors but cannot show a trend.


In [ ]:
run_nonlinear_convergence = True
N_values_nonlinear = [8, 12, 20, 30, 48]
newton_atol_nonlinear_convergence = 1e-10
nonlinear_convergence_results = ()
if run_nonlinear_convergence:
    nonlinear_convergence_results = tuple(
        run_defined_problem(
            nonlinear_problem, material=material, case=nonlinear_case,
            model="manufactured_nonlinear", N=N,
            newton_atol=newton_atol_nonlinear_convergence,
            accuracy=accuracy, verbosity=verbosity,
        )
        for N in N_values_nonlinear
    )
    print(report_convergence_table(nonlinear_convergence_results))
else:
    print("Nonlinear resolution study not run. Set run_nonlinear_convergence=True to enable it.")


## Why a flux-error peak can remain near an interface

Both manufactured problems share the same exact fields. Their linear and
nonlinear error curves can therefore be close once Newton error is small.
Increasing `N` improves the spatial approximation. Tightening Newton tolerance
alone may produce the same coefficients if the accepted Newton step already
falls below both targets.

For finite `N`, the global rational fields $u_N,w_N$ are smooth at every
material interface. Define $s_N=u_N'+\delta^2w_N'$. This quantity is continuous,
so its computed flux has the jump
$$
J_N(\xi^+)-J_N(\xi^-)
 =\bigl(\epsilon(\xi^+)-\epsilon(\xi^-)\bigr)s_N(\xi).
$$
The exact physical flux is continuous. Consequently, the finite-space flux
cannot generally match both exact traces at a nonzero-flux interface. A narrow
pointwise error peak can remain large while the physical $L^2$ error decreases.
Read the convergence tables together with the pointwise plots.

The appropriate accuracy target depends on the quantity needed for the study.
Accurate pointwise interface fluxes would require an interface-aware
approximation or a suitable flux reconstruction. Those are possible future
method changes; the cells here evaluate the existing mixed Galerkin solution.


## 8. Optional: compare two spatial resolutions

Set `run_resolution_check=True` and choose `N_fine` explicitly, greater
than `N_selected`. Set `newton_atol_fine` explicitly for the finer solve.
Using the same Newton target as the coarse solve helps isolate resolution
sensitivity. The relative changes refer to physical $L^2$ field norms. They are a
sensitivity check, **not a rigorous error bound**. More interfaces, narrow
layers, and large contrasts can require additional resolution.


In [ ]:
run_resolution_check = True
N_fine = 64
newton_atol_fine = 1e-10
resolution_check = None
if run_resolution_check:
    resolution_check = compare_resolution(
        experiment, fine_N=N_fine, newton_atol=newton_atol_fine, verbosity=verbosity,
    )
    print(report_experiment(experiment, resolution=resolution_check))
else:
    print("Resolution comparison not run. Set run_resolution_check=True to enable it.")

notebook_cases["selected"] = CaseArtifacts(
    "selected", experiment, figures=selected_figures, resolution=resolution_check,
)


## 9. Optional: mpmath arithmetic

Set `N_high_precision` and `newton_atol_high_precision` beside this separate
graded one-interface example. The initial Newton target is `"1e-35"`.
Its profile uses 50-digit arithmetic independently of `N_high_precision`. It does
not certify 50 digits in the spatial approximation or numerical integration.
Use resolution and integration checks separately. All internal evaluations
remain in the selected arithmetic; only plot data are converted to float64.


In [ ]:
run_high_precision_example = True
N_high_precision = 50
newton_atol_high_precision = 1e-35
high_precision_result = None
if run_high_precision_example:
    high_precision_material = preset_material(1, kind="graded")
    high_precision_problem, high_precision_case = define_nonlinear_problem(
        high_precision_material, delta=delta, beta=beta_nonlinear,
    )
    high_precision_result = run_defined_problem(
        high_precision_problem, material=high_precision_material, case=high_precision_case,
        model="manufactured_nonlinear", N=N_high_precision,
        newton_atol=newton_atol_high_precision,
        accuracy="high_precision", verbosity=verbosity,
    )
    print(report_experiment(high_precision_result))
    print(report_verification(high_precision_result))
    plot_errors(high_precision_result, x_min=plot_x_min, x_max=plot_x_max)
    plt.show()
else:
    print("Separate high-precision example not run. The main accuracy setting can also select it.")


## 10. Optional advanced material: an arbitrary piecewise-smooth function

A layer table is sufficient for the main workflow. This optional example shows
how a researcher can supply a different positive function inside each region.
Declare every interface, bounds valid everywhere, and the right wall trace.
These declarations are not inferred from a plot. The solver needs no derivative
of this function. Use the supplied arithmetic `a` for mpmath compatibility.

Custom executable functions must be supplied again when reloading a saved
solution. Known-answer checks use layer tables, while BSK accepts this
custom coefficient directly.


In [ ]:
run_custom_function_example = True
N_custom = 32
newton_atol_custom = 1e-10
custom_result = None

def custom_epsilon(x, a):
    if x < a.scalar("1"):
        return 1 + a.scalar("0.1")*a.exp(-x)
    if x < a.scalar("3"):
        return 2 + a.scalar("0.1")*a.exp(-(x-1))
    return a.scalar("1.5") + a.scalar("0.1")*a.exp(-(x-3))

custom_coefficient = Coefficient(
    custom_epsilon, lower_bound="1", upper_bound="2.1", wall_value="1.1",
    interfaces=("1", "3"),
)
if run_custom_function_example:
    custom_problem = define_bsk_problem(
        custom_coefficient, wall_potential=wall_potential,
        delta=delta, gamma=gamma, sigma=sigma, mu=mu,
    )
    custom_result = run_defined_problem(
        custom_problem, material=custom_coefficient, case=None, model="BSK",
        N=N_custom, newton_atol=newton_atol_custom,
        accuracy=accuracy, verbosity=verbosity,
    )
    print(report_experiment(custom_result))
    plot_experiment(custom_result, x_min=plot_x_min, x_max=plot_x_max)
    plt.show()
else:
    print("Custom-function solve not run. Its definition is ready for optional use.")


## 11. Optional: solve all presets

This explicitly runs **18 cases**: all six interface counts with BSK and both
known-answer RHS choices, with the selected physical inputs, material style, and accuracy. Start
with standard arithmetic. A high-precision sweep costs more; it does not run
unless enabled. The local factory below reuses the visible physical definitions
for each material. The editable `N_by_interfaces` table explicitly selects the
resolution for each interface count; all three source models use that row's `N`.
Set `newton_atol_sweep` explicitly for all cases in this sweep.
The initial table preserves the demonstration sizes and does not certify
adequate resolution for every material. Successful rows verify equations and
integration; use refinement and known-answer errors to assess spatial accuracy.


In [ ]:
run_all_cases = True
N_by_interfaces = {
    1: 12,
    2: 12,
    3: 12,
    5: 12,
    7: 12,
    10: 12,
}
newton_atol_sweep = 1e-10
sweep_results = []
def define_sweep_problems(sweep_material):
    sweep_bsk = define_bsk_problem(
        sweep_material.coefficient, wall_potential=wall_potential,
        delta=delta, gamma=gamma, sigma=sigma, mu=mu,
    )
    sweep_linear, sweep_linear_case = define_linear_problem(sweep_material, delta=delta)
    sweep_nonlinear, sweep_nonlinear_case = define_nonlinear_problem(
        sweep_material, delta=delta, beta=beta_nonlinear,
    )
    return {
        "BSK": (sweep_bsk, None),
        "manufactured_linear": (sweep_linear, sweep_linear_case),
        "manufactured_nonlinear": (sweep_nonlinear, sweep_nonlinear_case),
    }

if run_all_cases:
    for count, resolution in N_by_interfaces.items():
        sweep_material = preset_material(count, kind=material_style)
        for sweep_model, (sweep_problem, sweep_case) in define_sweep_problems(sweep_material).items():
            current = run_defined_problem(
                sweep_problem, material=sweep_material, case=sweep_case, model=sweep_model,
                N=resolution, newton_atol=newton_atol_sweep, accuracy=accuracy, verbosity=0,
            )
            sweep_results.append(current)
            print(report_experiment(current))
            print()
else:
    print("All-preset sweep not run. All six material profiles are shown in the gallery above.")


## 11a. Optional: adaptive homotopy recovery from a difficult initial guess

This separate experiment has its own editable material and physical parameters;
changing the main layers does not change this demonstration. Set
`run_homotopy_demo=True` below to enable its three runs. The definition and all
numerical budgets are visible. NumPy is selected initially; set
`hard_use_mpmath=True` for native mpmath arithmetic at `hard_precision` digits.

The source is the same manufactured family with a stronger cubic reaction:
$$
f(x,u)=J_*'(x)+(u-u_*)+10(u^3-u_*^3),\qquad f_u=1+30u^2.
$$
The requested initial guess is $10e^{-x^2}$; the Galerkin projection enforces
the prescribed wall value $u(0)=1$. The other wall datum is $u'''(0)=-6$.
This is a difficult starting field, rather than a change to the exact reference.

The comparison is a **short-stage-budget recovery demonstration**. Direct
Newton also succeeds when given a larger budget. The example does not establish
that homotopy is necessary or faster. Numerical convergence and known-answer
field accuracy remain separate checks, especially for the interface flux.


In [ ]:
run_homotopy_demo = True

hard_layers = [("0", "1"), ("1", "2")]
hard_material = LayeredMaterial(hard_layers)
hard_delta = "1"
hard_beta = "10"
hard_initial_amplitude = "10"
N_hard = 8
hard_L = "1"
newton_atol_hard = "1e-9"
hard_stage_iterations = 5
hard_control_iterations = 35
hard_use_mpmath = False
hard_precision = 30
hard_quadrature_atol = "1e-8"
hard_quadrature_rtol = "1e-8"


def define_hard_problem(material, *, delta, beta, initial_amplitude):
    reference = layered_transmission_case(material, delta=delta, nonlinear=True, beta=beta)

    def f(x, u, a):
        exact = reference.u(x, a)
        return (reference.flux_dx(x, a) + (u - exact)
                + a.scalar(beta) * (u**3 - exact**3))

    def f_u(x, u, a):
        return a.scalar(1) + 3 * a.scalar(beta) * u**2

    def initial_guess(x, a):
        return a.scalar(initial_amplitude) * a.exp(-x*x)

    problem = FluxProblem(
        delta=delta, epsilon=material.coefficient, domain=(0, "inf"),
        f=f, f_u=f_u,
        boundary_conditions=(
            ("1", None, None, "-6", None),
            ("0", None, None, None, None),
        ),
        initial_guess=initial_guess,
    )
    return problem, reference


hard_problem, hard_case = define_hard_problem(
    hard_material, delta=hard_delta, beta=hard_beta,
    initial_amplitude=hard_initial_amplitude,
)
hard_continuation = ContinuationOptions(
    reference=ScreenedLinearReference(strength="1"),
    initial_step="0.5", max_step="0.5", min_step="1e-6", growth="1.5",
    max_stages=40, max_total_iterations=150,
)
hard_numerical = dict(
    L=hard_L, use_mpmath=hard_use_mpmath, precision=hard_precision,
    quadrature_order=2*N_hard+4,
    quadrature_atol=hard_quadrature_atol, quadrature_rtol=hard_quadrature_rtol,
    max_refinements=2, newton_fallback=None, continuation=hard_continuation,
)


## 11b. Run the three comparisons

All runs receive the **same** `hard_problem`, resolution, starting field,
arithmetic, and absolute residual target. Newton backtracking is active in
all three; adjoint recovery is disabled to keep this comparison focused.

- `direct_limited`: direct Newton with the short iteration budget.
- `homotopy`: the same per-stage budget, followed by reference continuation
  when needed. Its total-work budget includes the initial physical attempt
  and all accepted or rejected continuation stages.
- `direct_control`: direct Newton with the larger, explicit control budget.

For the calibrated settings, the first run stops after five Newton attempts.
The homotopy run and the larger-budget control converge. The controller's
reference linear solves are additional work and are listed separately from
Newton attempts; the runs do not have equal total-work allowances.


In [ ]:
hard_results = {}
if run_homotopy_demo:
    for hard_label, hard_globalization, hard_iteration_limit in (
        ("direct_limited", False, hard_stage_iterations),
        ("homotopy", True, hard_stage_iterations),
        ("direct_control", False, hard_control_iterations),
    ):
        print("Running:", hard_label)
        hard_results[hard_label] = run_defined_problem(
            hard_problem, material=hard_material, case=hard_case,
            model="manufactured_nonlinear", N=N_hard, newton_atol=newton_atol_hard,
            accuracy="standard", verbosity=verbosity,
            numerical=dict(hard_numerical, max_iterations=hard_iteration_limit,
                           use_globalization=hard_globalization),
        )
    print(report_homotopy_comparison(hard_results))
else:
    print("Homotopy demonstration not run. Set run_homotopy_demo=True to enable it.")


## 11c. Inspect the reference, trial stages, and rollback

Continuation uses the complete residual blend
$$
R_h(z;q)=(1-q)R_h^{\rm physical}(z)+qR_h^{\rm reference}(z).
$$
The physical callbacks and boundary conditions retain their definitions.
`q=1` is the verified reference and `q=0` is the original physical problem.
Stage residuals at other values belong to their blended equations.

For the initial settings, a trial at `q=0.5` misses the Newton target. The
controller keeps the accepted reference state at `q=1`, reduces the step,
and accepts `q=0.75`, then `q=0.375` and `q=0`. The actual table below shows
what happened on each quadrature plan; changing inputs may change this path.
The initial physical attempt is identified separately from continuation.


In [ ]:
hard_stage_records = ()
if run_homotopy_demo:
    hard_recovered = hard_results["homotopy"]
    hard_stage_records = homotopy_stage_rows(hard_recovered)
    print(report_homotopy_stages(hard_recovered))
    hard_reference_used = any(
        row["kind"] == "reference" and row["accepted"] for row in hard_stage_records
    )
    hard_positive_stage = any(
        row["kind"] == "continuation" and row["accepted"] and 0 < row["q"] < 1
        for row in hard_stage_records
    )
    if hard_recovered.solution.converged and hard_reference_used and hard_positive_stage:
        print("Reference continuation was used and the original physical problem is verified at q=0.")
    elif hard_recovered.solution.converged:
        print("The physical problem converged without needing the expected continuation path.")
    else:
        print("The physical problem is not verified. Inspect the stage reasons and work limits.")


## 11d. Read the path, residuals, and recovered fields

The parameter plot distinguishes accepted states from rejected trials. A
rejected trial does not replace the previously accepted state. The residual
plot uses each stage's own target, so separate blended equations remain
identifiable. Reference points are shown explicitly.

The field comparison marks the short-budget result as unverified. It also
shows the recovered result, the larger-budget Newton control, and the known
reference. Nearly overlapping converged fields indicate agreement on the
same finite-dimensional problem. Use the known-answer error table to assess
spatial accuracy; a small final Newton residual does not remove that error.


In [ ]:
hard_demo_figures = {}
if run_homotopy_demo:
    hard_demo_figures["parameter-path"] = plot_homotopy_path(hard_results["homotopy"])
    hard_demo_figures["residuals"] = plot_convergence(hard_results['homotopy'], iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
    hard_demo_figures["fields"] = plot_homotopy_comparison(hard_results, x_min=plot_x_min, x_max=plot_x_max)
    hard_demo_figures["known-answer-errors"] = plot_errors(hard_results['homotopy'], x_min=plot_x_min, x_max=plot_x_max)
    plt.show()
    print("Homotopy result:")
    print(report_error_table(hard_results["homotopy"]))
    print("Larger-budget direct Newton control:")
    print(report_error_table(hard_results["direct_control"]))


## 12. Save and share the case bundles

Each registered case has its own result, figures, and output folder. The selected
case also owns the material previews and its optional resolution comparison.
The larger and extreme BSK cases are included only when their own switches are
enabled and their results exist. The linear and nonlinear checks keep their own
error figures. Changing the main `model` does not redirect these extra cases.

Every folder contains `settings.json`, a typed `solution.json`, and
`profiles.csv`, plus that case's PDF figures. Decimal values retain native
precision in the CSV. Samples exactly at interfaces use right traces. Settings
record the actual problem, N, Newton tolerance, and verification status; no
executable callbacks are serialized. Figures are checked against the solution
they describe before export.


In [ ]:
save_results = True
saved_files = {}
notebook_cases["selected"] = CaseArtifacts(
    "selected", experiment, figures=selected_figures, resolution=resolution_check,
)
if run_large_bsk and bsk_large_experiment is not None:
    notebook_cases["bsk-large"] = CaseArtifacts(
        "bsk-large", bsk_large_experiment, figures=bsk_large_figures,
    )
else:
    notebook_cases.pop("bsk-large", None)
if run_extreme_bsk_mpmath and bsk_extreme_experiment is not None:
    notebook_cases["bsk-extreme"] = CaseArtifacts(
        "bsk-extreme", bsk_extreme_experiment, figures=bsk_extreme_figures,
    )
else:
    notebook_cases.pop("bsk-extreme", None)
if save_results:
    saved_files = export_cases(notebook_cases, output_directory)
    print("Saved case bundles under:", output_directory)
    for case_name, paths in saved_files.items():
        print(case_name, "->", paths["settings"].parent)
else:
    print("No files written. Set save_results=True to export the registered case bundles.")


## Numerical details and further work

Every solve requires explicit `N` and `newton_atol`; the profiles never change
these choices. Standard
and strict-integration profiles use NumPy. High precision uses mpmath with an
independent 50-digit context. Arithmetic precision and integration tolerances
remain separate from spatial resolution. Reports and figure titles show `N`
and the actual unknown count.

The automatic starting quadrature order is `2*N+4` points per integration panel;
integration is checked independently and may refine further. Advanced users can
pass an explicit order and other solver settings in the `numerical` dictionary.
`N` and `newton_atol` always belong in their required arguments, outside that
dictionary. The numerical wrapper maps `newton_atol` to the public `solve`
function's `tolerance` argument and passes the explicitly supplied problem unchanged. Reports show the requested target and the achieved residual norm.

The manufactured family starts from $1/(1+x)$ and adds analytic cubic-onset
terms at interfaces, chosen to preserve physical flux. Its reference and RHS
are independent of the numerical Galerkin coefficients. Smooth pieces of
$\epsilon$ are differentiated only inside this verification construction.

The solver uses a global conforming basis. It splits quadrature at every
interface but does not create a separate approximation mesh in each material.
Physical norms and weak residuals are more useful validation measures than
requiring an exact numerical flux trace at a single interface.

See [numerical methods](../../docs/numerical_methods.md) and
[installation](../../INSTALL.md) for the package interfaces and environment.
